In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# MILESTONE - 1

**This milestone builds a foundational text-processing pipeline using standard Python libraries and scikit-learn to establish a performance benchmark. It involves cleaning dataset text by removing punctuation and stop words, converting data into statistical feature matrices using TF-IDF tokenization, and measuring similarity scores via cosine distance. Finally, a strict MAP@3 evaluation script was implemented to test initial baseline prediction models before moving onto deep-learning transformers.**

In [2]:
import pandas as pd
import string
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Simply load train and test data directly by path
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Done! Datasets loaded successfully.")

Done! Datasets loaded successfully.


In [3]:
# Q1. Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts,            what is the sum of the occurrences of the most frequent option and the least frequent option?  

# Calculate the frequency distribution of correct answers
counts = train_df['answer'].value_counts()
print("Frequency Distribution")
print(counts)

# Get the max (most frequent) and min (least frequent) occurrences
most_frequent_count = counts.max()
least_frequent_count = counts.min()

most_frequent_option = counts.idxmax()
least_frequent_option = counts.idxmin()

# Calculate the sum
final_sum = most_frequent_count + least_frequent_count

print(f"Most frequent: Option {most_frequent_option} ({most_frequent_count} times)")
print(f"Least frequent: Option {least_frequent_option} ({least_frequent_count} times)")
print(f"Sum of occurrences: {final_sum}")

Frequency Distribution
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Most frequent: Option B (490 times)
Least frequent: Option E (324 times)
Sum of occurrences: 814


Q1. Sum of occurrences: 814


In [4]:
# Q2. After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

# Initialize an empty set to store all unique words
vocabulary_set = set()

# Loop through each prompt in the globally loaded train_df
for prompt in train_df['prompt'].dropna().astype(str):
    # 1. Convert to lowercase
    cleaned_prompt = prompt.lower()
    
    # 2. Remove standard punctuation characters using string.punctuation
    translator = str.maketrans('', '', string.punctuation)
    cleaned_prompt = cleaned_prompt.translate(translator)
    
    # 3. Split the text by whitespace into words
    words = cleaned_prompt.split()
    
    # Add words to our set to maintain uniqueness
    vocabulary_set.update(words)

# Calculate total number of unique words
total_unique_words = len(vocabulary_set)

print(f"Total number of unique words (Vocabulary Size): {total_unique_words}")

Total number of unique words (Vocabulary Size): 859


Q2. Total number of unique words (Vocabulary Size): 859


In [5]:
# Q3. Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?  

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# 1. Fetch prompt for Row ID 1 (which corresponds to row index where id == 1)
row_1_prompt = train_df.loc[train_df['id'] == 1, 'prompt'].values[0]

# 2. Clean text: convert to lowercase and translate out punctuation characters
cleaned_text = row_1_prompt.lower()
translator = str.maketrans('', '', string.punctuation)
cleaned_text = cleaned_text.translate(translator)

# 3. Split by whitespace into a list of words
all_words = cleaned_text.split()

# 4. Filter out standard English stop words using sklearn's set
filtered_words = [word for word in all_words if word not in ENGLISH_STOP_WORDS]

# Count the remaining words
words_remaining = len(filtered_words)

print(f"Original Text: {row_1_prompt}")
print(f"Words remaining after stop word removal: {filtered_words}")
print(f"Number of words left: {words_remaining}")

Original Text: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Words remaining after stop word removal: ['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Number of words left: 13


Q3. Number of words left: 13


In [6]:
# Q4. Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

# 1. Gather all unique text blocks into a single flat list
# This includes the text from the prompt, and options A, B, C, D, and E across all rows
combined_text_list = []

for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    # Drop any nulls, convert to standard strings, and add to our text list
    combined_text_list.extend(train_df[col].dropna().astype(str).tolist())

# 2. Initialize the default TfidfVectorizer with English stop words
vectorizer = TfidfVectorizer(stop_words='english')

# 3. Fit the vectorizer on the combined list of texts
vectorizer.fit(combined_text_list)

# 4. Get the total number of feature columns (vocabulary size)
total_features = len(vectorizer.get_feature_names_out())

print(f"Total text blocks combined: {len(combined_text_list)}")
print(f"Exact total number of feature columns: {total_features}")

Total text blocks combined: 12000
Exact total number of feature columns: 2762


Q4. Exact total number of feature columns: 2762


In [7]:
# Q5. Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

# 1. Ensure the vectorizer from Q4 is fitted on the combined text
combined_text_list = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    combined_text_list.extend(train_df[col].dropna().astype(str).tolist())

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text_list)

# 2. Extract the prompt and option A for Row ID 1
row_1_data = train_df[train_df['id'] == 1].iloc[0]
prompt_text = str(row_1_data['prompt'])
option_A_text = str(row_1_data['A'])

# 3. Transform the text into TF-IDF vectors
prompt_vector = vectorizer.transform([prompt_text])
option_A_vector = vectorizer.transform([option_A_text])

# 4. Calculate cosine similarity
similarity_score = cosine_similarity(prompt_vector, option_A_vector)[0][0]

# 5. Round to 4 decimal places
rounded_score = round(similarity_score, 4)

print(f"Prompt: {prompt_text[:100]}...")
print(f"Option A: {option_A_text}")
print(f"Cosine Similarity Score (4 decimal places): {rounded_score:.4f}")

Prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and ...
Option A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time.
Cosine Similarity Score (4 decimal places): 0.2328


In [8]:
# Q6. Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.   

# 1. Re-initialize and fit the vectorizer on all text columns combined (from Q4 logic)
combined_text_list = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    combined_text_list.extend(train_df[col].dropna().astype(str).tolist())

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text_list)

# 2. Track standard metrics
correct_matches = 0
total_rows = len(train_df)

# List of option keys
options_keys = ['A', 'B', 'C', 'D', 'E']

# 3. Iterate through every single row in the dataset
for idx, row in train_df.iterrows():
    prompt_text = str(row['prompt'])
    correct_ans = str(row['answer']).strip()
    
    # Transform the prompt text into its TF-IDF vector
    prompt_vector = vectorizer.transform([prompt_text])
    
    highest_similarity = -1.0
    best_matching_option = None
    
    # Check each option (A, B, C, D, E) for this row
    for opt in options_keys:
        option_text = str(row[opt])
        option_vector = vectorizer.transform([option_text])
        
        # Calculate cosine similarity score
        sim_score = cosine_similarity(prompt_vector, option_vector)[0][0]
        
        # Track the option with the highest score
        if sim_score > highest_similarity:
            highest_similarity = sim_score
            best_matching_option = opt
            
    # Check if the highest similarity option matches the ground truth answer
    if best_matching_option == correct_ans:
        correct_matches += 1

# 4. Calculate the percentage match accuracy
matching_percentage = (correct_matches / total_rows) * 100

print("\n Summary Verification ")
print(f"Total Rows Evaluated: {total_rows}")
print(f"Correct Highest Similarity Matches: {correct_matches}")
print("\n--- Final Answer for Form ---")
print(f"Percentage of correct instances: {matching_percentage:.2f}%")


 Summary Verification 
Total Rows Evaluated: 2000
Correct Highest Similarity Matches: 274

--- Final Answer for Form ---
Percentage of correct instances: 13.70%


Q6. Percentage of correct instances: 13.70%
Q7. Final MAP@3 score of 1.0
Q8. On the basis of previous (Q7) explanation answer is 0.5

In [9]:
# Q9. The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

# 1. Get the frequency distribution of correct answers and find the ordered top 3 options
answer_counts = train_df['answer'].value_counts()
top_3_options = list(answer_counts.index[:3])

for i, opt in enumerate(top_3_options, 1):
    print(f"Rank {i} Guess: Option {opt} ({answer_counts[opt]} occurrences)")

# 2. Extract our static guesses
top_1, top_2, top_3 = top_3_options

# 3. Calculate MAP@3 score for each row based on this static choice
total_map3_score = 0.0
total_rows = len(train_df)

for idx, row in train_df.iterrows():
    actual_answer = str(row['answer']).strip()
    
    # Calculate score based on position of actual answer in static prediction [top_1, top_2, top_3]
    if actual_answer == top_1:
        total_map3_score += 1.0      # Rank 1 match = 1/1
    elif actual_answer == top_2:
        total_map3_score += 0.5      # Rank 2 match = 1/2
    elif actual_answer == top_3:
        total_map3_score += 1.0 / 3.0 # Rank 3 match = 1/3
    else:
        total_map3_score += 0.0      # Not in top 3 predictions

# 4. Compute overall Mean Average Precision
overall_map3 = total_map3_score / total_rows

print(f"Overall MAP@3 Score: {overall_map3:.4f}")

Rank 1 Guess: Option B (490 occurrences)
Rank 2 Guess: Option C (459 occurrences)
Rank 3 Guess: Option A (369 occurrences)
Overall MAP@3 Score: 0.4213


Q9. Overall MAP@3 Score: 0.4213


In [10]:
# Q10. The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  

# 1. Re-fit the vectorizer on all text combined (from Q4 logic)
combined_text_list = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    combined_text_list.extend(train_df[col].dropna().astype(str).tolist())

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(combined_text_list)

# Options list
options_keys = ['A', 'B', 'C', 'D', 'E']

total_map3_score = 0.0
total_rows = len(train_df)

# 2. Iterate through every single row to make ranked top 3 predictions
for idx, row in train_df.iterrows():
    prompt_text = str(row['prompt'])
    correct_ans = str(row['answer']).strip()
    
    # Vectorize prompt
    prompt_vector = vectorizer.transform([prompt_text])
    
    # Store option labels and their corresponding cosine similarities
    option_similarities = []
    for opt in options_keys:
        option_text = str(row[opt])
        option_vector = vectorizer.transform([option_text])
        sim_score = cosine_similarity(prompt_vector, option_vector)[0][0]
        option_similarities.append((opt, sim_score))
        
    # 3. Sort options from highest similarity score to lowest
    # To resolve ties cleanly, python's Timsort preserves index order stability
    option_similarities.sort(key=lambda x: x[1], reverse=True)
    
    # Take the top 3 ranked options
    top_3_predictions = [item[0] for item in option_similarities[:3]]
    
    # 4. Calculate MAP@3 score for this individual row
    if correct_ans in top_3_predictions:
        rank_idx = top_3_predictions.index(correct_ans)
        if rank_idx == 0:
            total_map3_score += 1.0       # Rank 1 match -> 1/1
        elif rank_idx == 1:
            total_map3_score += 0.5       # Rank 2 match -> 1/2
        elif rank_idx == 2:
            total_map3_score += 1.0 / 3.0  # Rank 3 match -> 1/3
    else:
        total_map3_score += 0.0           # Not in top 3 choices

# 5. Compute the final average MAP@3 score
final_average_map3 = total_map3_score / total_rows

print(f"Final Average MAP@3 Score of Pipeline: {final_average_map3:.4f}")

Final Average MAP@3 Score of Pipeline: 0.3119


Q10. Final Average MAP@3 Score of Pipeline: 0.3119
